# Hybrid Museum Recommendation EngineThis notebook implements **hybrid recommendation strategies** that combine:- **Content-Based**: Sentence Transformers (semantic understanding)- **Graph-Based**: Node2Vec (structural relationships)## The ChallengeWhen content and graph similarity distributions **don't overlap**, simple averaging dilutes strong signals from each model. We need smarter fusion strategies.| Strategy | Formula | Use Case ||----------|---------|----------|| **Max** | `max(sim_c, sim_g)` | Recommend if ANY model finds relevant || **Rank Fusion** | Combine ranks, not scores | Scores not comparable || **Contextual** | Choose model by context | Use graph for artist-rich museums || **Intersection** | Recommend if BOTH agree | High precision |---## 1. Loading Models

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics.pairwise import cosine_similarityfrom sklearn.preprocessing import normalizeimport pickleimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8-whitegrid')# Load modelsprint("Loading models...")df_content = pd.read_pickle('../outputs/df.pkl')X_content = np.stack(df_content['embedding_themes'].values)df_graph = pd.read_pickle('../outputs/df_graph.pkl')with open('../outputs/museum_mapping.pkl', 'rb') as f:    mapping = pickle.load(f)# Align datasetscommon_ids = sorted(set(df_content['Identifiant']) & set(df_graph['Identifiant']))content_idx = {mid: i for i, mid in enumerate(df_content['Identifiant'])}graph_idx = mapping['museum_to_idx']aligned_content, aligned_graph, metadata = [], [], []for mid in common_ids:    aligned_content.append(X_content[content_idx[mid]])    aligned_graph.append(df_graph[df_graph['Identifiant'] == mid]['embedding_graph'].iloc[0])    row = df_content.iloc[content_idx[mid]]    metadata.append({        'id': mid, 'name': row.get('Nom officiel', mid),        'region': row.get('Région', 'Unknown'), 'themes': row.get('Thèmes', '')    })X_content_aligned = normalize(np.array(aligned_content))X_graph_aligned = normalize(np.array(aligned_graph))# Compute base similaritiessim_content = cosine_similarity(X_content_aligned)sim_graph = cosine_similarity(X_graph_aligned)print(f"✓ {len(common_ids)} museums aligned")print(f"✓ Content: {X_content_aligned.shape}, Graph: {X_graph_aligned.shape}")

---## 2. Distribution AnalysisFirst, let's visualize why simple averaging is problematic.

In [ ]:
n = len(metadata)upper_content = sim_content[np.triu_indices(n, k=1)]upper_graph = sim_graph[np.triu_indices(n, k=1)]fig, axes = plt.subplots(1, 3, figsize=(16, 5))# Histogramsax = axes[0]ax.hist(upper_content, bins=50, alpha=0.6, label=f'Content (μ={upper_content.mean():.3f})', color='steelblue', density=True)ax.hist(upper_graph, bins=50, alpha=0.6, label=f'Graph (μ={upper_graph.mean():.3f})', color='coral', density=True)ax.set_xlabel('Cosine Similarity')ax.set_ylabel('Density')ax.set_title('Similarity Distributions')ax.legend()# Scatter plotax = axes[1]sample_idx = np.random.choice(len(upper_content), min(10000, len(upper_content)), replace=False)ax.scatter(upper_content[sample_idx], upper_graph[sample_idx], alpha=0.1, s=1, c='purple')ax.plot([0, 1], [0, 1], 'r--', label='Perfect correlation')ax.set_xlabel('Content Similarity')ax.set_ylabel('Graph Similarity')ax.set_title(f'Correlation: r={np.corrcoef(upper_content, upper_graph)[0,1]:.3f}')ax.legend()# Problem illustrationax = axes[2]# Show what averaging doesupper_avg = (upper_content + upper_graph) / 2ax.hist(upper_content, bins=50, alpha=0.4, label='Content', color='steelblue', density=True)ax.hist(upper_graph, bins=50, alpha=0.4, label='Graph', color='coral', density=True)ax.hist(upper_avg, bins=50, alpha=0.6, label='Average (diluted!)', color='green', density=True)ax.set_xlabel('Cosine Similarity')ax.set_title('Problem: Averaging Dilutes Signals')ax.legend()plt.tight_layout()plt.savefig('../outputs/distribution_analysis.png', dpi=150, bbox_inches='tight')plt.show()# Overlap quantificationoverlap_threshold = 0.1content_high = upper_content > (upper_content.mean() + upper_content.std())graph_high = upper_graph > (upper_graph.mean() + upper_graph.std())both_high = content_high & graph_highprint(f"\nDistribution Analysis:")print(f"  Content mean: {upper_content.mean():.4f}, std: {upper_content.std():.4f}")print(f"  Graph mean: {upper_graph.mean():.4f}, std: {upper_graph.std():.4f}")print(f"  Correlation: {np.corrcoef(upper_content, upper_graph)[0,1]:.4f}")print(f"  High in BOTH models: {both_high.sum()/len(both_high)*100:.1f}% of pairs")

---## 3. Hybrid Fusion Strategies### Strategy 1: Max FusionRecommend if **at least one** model finds the pair similar.

In [ ]:
def hybrid_max(sim_content, sim_graph):    """Take maximum similarity from either model."""    return np.maximum(sim_content, sim_graph)sim_max = hybrid_max(sim_content, sim_graph)upper_max = sim_max[np.triu_indices(n, k=1)]print("Max Fusion:")print(f"  Mean: {upper_max.mean():.4f} (vs content {upper_content.mean():.4f}, graph {upper_graph.mean():.4f})")print(f"  → Captures strong signals from EITHER model")

### Strategy 2: Reciprocal Rank Fusion (RRF)Combine **rankings** instead of scores. This handles different score scales.

In [ ]:
def reciprocal_rank_fusion(sim_content, sim_graph, k=60):    """    RRF: score = sum(1 / (k + rank_i))    Higher k = less emphasis on top ranks    """    n = sim_content.shape[0]    rrf_matrix = np.zeros((n, n))        for i in range(n):        # Get ranks for each model (descending similarity)        ranks_content = np.argsort(np.argsort(-sim_content[i]))        ranks_graph = np.argsort(np.argsort(-sim_graph[i]))                # RRF score        rrf_scores = 1/(k + ranks_content) + 1/(k + ranks_graph)        rrf_matrix[i] = rrf_scores        # Symmetrize    rrf_matrix = (rrf_matrix + rrf_matrix.T) / 2    return rrf_matrixsim_rrf = reciprocal_rank_fusion(sim_content, sim_graph)upper_rrf = sim_rrf[np.triu_indices(n, k=1)]print("Reciprocal Rank Fusion:")print(f"  Mean RRF score: {upper_rrf.mean():.6f}")print(f"  → Combines rankings, not raw scores")

### Strategy 3: Contextual SwitchingUse **graph** for museums with rich artist/figure data, **content** otherwise.

In [ ]:
# Load original data to check artist presencedf_full = pd.read_csv('../musees-de-france-base-museofile.csv', sep=';')def has_rich_metadata(museum_id):    """Check if museum has artist or figure data."""    row = df_full[df_full['Identifiant'] == museum_id]    if len(row) == 0:        return False    artist = row.iloc[0].get('Artiste')    figure = row.iloc[0].get('Personnage phare')    has_artist = pd.notna(artist) and len(str(artist)) > 5    has_figure = pd.notna(figure) and len(str(figure)) > 5    return has_artist or has_figure# Create mask for museums with rich metadatarich_metadata = np.array([has_rich_metadata(m['id']) for m in metadata])print(f"Museums with rich metadata: {rich_metadata.sum()}/{len(rich_metadata)} ({rich_metadata.mean()*100:.1f}%)")def hybrid_contextual(sim_content, sim_graph, rich_mask):    """    For museum pairs where BOTH have rich metadata: use graph    Otherwise: use content    """    n = sim_content.shape[0]    sim_contextual = np.zeros((n, n))        for i in range(n):        for j in range(n):            if rich_mask[i] and rich_mask[j]:                # Both have rich metadata: trust graph more                sim_contextual[i, j] = 0.3 * sim_content[i, j] + 0.7 * sim_graph[i, j]            elif rich_mask[i] or rich_mask[j]:                # One has rich metadata: balanced                sim_contextual[i, j] = 0.5 * sim_content[i, j] + 0.5 * sim_graph[i, j]            else:                # Neither has rich metadata: trust content                sim_contextual[i, j] = 0.8 * sim_content[i, j] + 0.2 * sim_graph[i, j]        return sim_contextualsim_contextual = hybrid_contextual(sim_content, sim_graph, rich_metadata)upper_contextual = sim_contextual[np.triu_indices(n, k=1)]print(f"\nContextual Fusion:")print(f"  Mean: {upper_contextual.mean():.4f}")print(f"  → Adapts weighting based on data availability")

### Strategy 4: Intersection (High Precision)Only recommend if **both** models agree.

In [ ]:
def hybrid_intersection(sim_content, sim_graph, threshold_percentile=90):    """    Geometric mean, but only for pairs where both scores are above threshold.    Others get score = 0.    """    thresh_c = np.percentile(sim_content[np.triu_indices(n, k=1)], threshold_percentile)    thresh_g = np.percentile(sim_graph[np.triu_indices(n, k=1)], threshold_percentile)        # Geometric mean where both are high    both_high = (sim_content >= thresh_c) & (sim_graph >= thresh_g)    sim_inter = np.where(both_high, np.sqrt(sim_content * sim_graph), 0)        return sim_inter, thresh_c, thresh_gsim_intersection, thresh_c, thresh_g = hybrid_intersection(sim_content, sim_graph, 80)upper_inter = sim_intersection[np.triu_indices(n, k=1)]print(f"Intersection Fusion (top 20% in both):")print(f"  Content threshold: {thresh_c:.4f}")print(f"  Graph threshold: {thresh_g:.4f}")print(f"  Non-zero pairs: {(upper_inter > 0).sum()} / {len(upper_inter)} ({(upper_inter > 0).mean()*100:.1f}%)")print(f"  → High precision, lower recall")

---## 4. Strategy Comparison

In [ ]:
strategies = {    'Content Only': sim_content,    'Graph Only': sim_graph,    'Average': (sim_content + sim_graph) / 2,    'Max': sim_max,    'RRF': sim_rrf,    'Contextual': sim_contextual,}fig, axes = plt.subplots(2, 3, figsize=(16, 10))axes = axes.flatten()for ax, (name, sim) in zip(axes, strategies.items()):    upper = sim[np.triu_indices(n, k=1)]    ax.hist(upper, bins=50, color='steelblue' if 'Content' in name else 'coral' if 'Graph' in name else 'green',             alpha=0.7, edgecolor='white')    ax.axvline(upper.mean(), color='red', linestyle='--', label=f'μ={upper.mean():.3f}')    ax.set_title(name, fontsize=12)    ax.set_xlabel('Similarity')    ax.legend()plt.tight_layout()plt.savefig('../outputs/strategy_comparison.png', dpi=150, bbox_inches='tight')plt.show()

---## 5. Evaluating RecommendationsCompare top-k recommendations across strategies.

In [ ]:
def get_top_k(sim_matrix, idx, k=10):    scores = sim_matrix[idx]    top_idx = np.argsort(scores)[::-1][1:k+1]    return set(top_idx), scores[top_idx]def evaluate_strategy(sim_matrix, name, k=10):    """Evaluate recommendation quality."""    # Compare with content and graph    overlap_content = []    overlap_graph = []        for i in range(len(metadata)):        top_strat, _ = get_top_k(sim_matrix, i, k)        top_content, _ = get_top_k(sim_content, i, k)        top_graph, _ = get_top_k(sim_graph, i, k)                overlap_content.append(len(top_strat & top_content) / k)        overlap_graph.append(len(top_strat & top_graph) / k)        return {        'name': name,        'overlap_content': np.mean(overlap_content),        'overlap_graph': np.mean(overlap_graph),        'unique_contribution': 1 - max(np.mean(overlap_content), np.mean(overlap_graph))    }# Evaluate all strategiesresults = []for name, sim in strategies.items():    if name not in ['Content Only', 'Graph Only']:        results.append(evaluate_strategy(sim, name))results_df = pd.DataFrame(results)print("Strategy Evaluation (Top-10 recommendations):\n")print(results_df.to_string(index=False))# Visualizefig, ax = plt.subplots(figsize=(10, 6))x = range(len(results_df))width = 0.35bars1 = ax.bar([i - width/2 for i in x], results_df['overlap_content'], width, label='Overlap with Content', color='steelblue')bars2 = ax.bar([i + width/2 for i in x], results_df['overlap_graph'], width, label='Overlap with Graph', color='coral')ax.set_ylabel('Overlap Ratio')ax.set_title('Strategy Overlap with Base Models')ax.set_xticks(x)ax.set_xticklabels(results_df['name'], rotation=15)ax.legend()ax.set_ylim(0, 1)plt.tight_layout()plt.savefig('../outputs/strategy_evaluation.png', dpi=150, bbox_inches='tight')plt.show()

---## 6. Recommendation Examples

In [ ]:
def show_recommendations(museum_idx, strategies_dict, k=5):    """Show recommendations from each strategy side by side."""    museum = metadata[museum_idx]    print(f"{'='*80}")    print(f"SOURCE: {museum['name']}")    print(f"Region: {museum['region']}")    print(f"Themes: {museum['themes'][:60]}...")    print(f"{'='*80}\n")        for name, sim in strategies_dict.items():        top_idx, scores = get_top_k(sim, museum_idx, k)        print(f"{name}:")        for i, (idx, score) in enumerate(zip(list(top_idx)[:k], scores[:k]), 1):            m = metadata[idx]            print(f"  {i}. {m['name'][:45]} ({score:.3f})")        print()# Pick diverse examplessample_indices = [50, 200, 500]for idx in sample_indices:    show_recommendations(idx, {        'Content': sim_content,        'Graph': sim_graph,        'Max': sim_max,        'RRF': sim_rrf,        'Contextual': sim_contextual    }, k=3)

---## 7. Final Hybrid Recommender Class

In [ ]:
class HybridRecommender:    STRATEGIES = ['max', 'rrf', 'contextual', 'intersection', 'average']        def __init__(self, strategy='rrf'):        if strategy not in self.STRATEGIES:            raise ValueError(f"Strategy must be one of {self.STRATEGIES}")                self.strategy = strategy        self.metadata = metadata                # Compute similarity matrix based on strategy        if strategy == 'max':            self.sim_matrix = hybrid_max(sim_content, sim_graph)        elif strategy == 'rrf':            self.sim_matrix = reciprocal_rank_fusion(sim_content, sim_graph)        elif strategy == 'contextual':            self.sim_matrix = hybrid_contextual(sim_content, sim_graph, rich_metadata)        elif strategy == 'intersection':            self.sim_matrix, _, _ = hybrid_intersection(sim_content, sim_graph)        else:  # average            self.sim_matrix = (sim_content + sim_graph) / 2        def recommend(self, museum_idx, k=5):        """Get top-k recommendations."""        scores = self.sim_matrix[museum_idx]        top_idx = np.argsort(scores)[::-1][1:k+1]                return [{            'rank': i+1,            'idx': idx,            'name': self.metadata[idx]['name'],            'region': self.metadata[idx]['region'],            'score': scores[idx],            'content_sim': sim_content[museum_idx, idx],            'graph_sim': sim_graph[museum_idx, idx]        } for i, idx in enumerate(top_idx)]        def explain(self, source_idx, target_idx):        """Explain why a recommendation was made."""        return {            'strategy': self.strategy,            'hybrid_score': self.sim_matrix[source_idx, target_idx],            'content_score': sim_content[source_idx, target_idx],            'graph_score': sim_graph[source_idx, target_idx],            'source_has_metadata': rich_metadata[source_idx],            'target_has_metadata': rich_metadata[target_idx]        }# Demoprint("Creating RRF Recommender...")recommender = HybridRecommender(strategy='rrf')recs = recommender.recommend(100, k=5)print(f"\nRecommendations for: {metadata[100]['name']}\n")for rec in recs:    print(f"{rec['rank']}. {rec['name'][:40]}")    print(f"   Content: {rec['content_sim']:.3f} | Graph: {rec['graph_sim']:.3f} | Hybrid: {rec['score']:.6f}")

---## 8. Save Best Model

In [ ]:
# Save RRF model (best for non-overlapping distributions)best_strategy = 'rrf'best_recommender = HybridRecommender(strategy=best_strategy)np.save('sim_hybrid.npy', best_recommender.sim_matrix)config = {    'strategy': best_strategy,    'n_museums': len(metadata),    'museum_ids': common_ids,    'strategies_available': HybridRecommender.STRATEGIES}with open('hybrid_config.pkl', 'wb') as f:    pickle.dump(config, f)print(f"Saved hybrid model:")print(f"  Strategy: {best_strategy}")print(f"  Museums: {len(metadata)}")print(f"  Files: sim_hybrid.npy, hybrid_config.pkl")

---## Conclusion### Key InsightWhen content and graph distributions **don't overlap**, simple averaging dilutes signals. Better strategies:| Strategy | Best For ||----------|----------|| **RRF** | General use, handles different scales || **Max** | Maximize recall (find anything relevant) || **Contextual** | When metadata quality varies || **Intersection** | High precision requirements |### Recommendation**RRF (Reciprocal Rank Fusion)** is the default choice because:1. Combines rankings, not raw scores2. Works regardless of score distribution shapes3. No hyperparameters to tune (except k=60)### Files Generated- `sim_hybrid.npy`: Hybrid similarity matrix- `hybrid_config.pkl`: Configuration- `distribution_analysis.png`: Distribution comparison- `strategy_comparison.png`: All strategies- `strategy_evaluation.png`: Overlap analysis